# 02 — KPI Proposal Review

Interactive test bench for the Domain/KPI node's prompt engineering.

**What this notebook is for:**
- Load a real `DataProfile` (from the Profiler node output)
- Assemble the exact system prompt + user content that `run_domain_kpi` sends
- Call the LLM live (Gemini or Mistral) and inspect the raw structured response
- Run the deterministic SQL-column validator against the output
- **Execute the generated SQL** against a real SQLite table and check it actually runs and returns sensible rows
- Iterate on the prompt (edit the constants in Section 2, re-run from there down) without touching `agent/domain_kpi.py` until you're happy

**What this notebook is *not* for:** this does not replace `run_domain_kpi` in the pipeline. Once a prompt version is validated here, port the constants back into `agent/domain_kpi.py` (or better — import directly, see note in Section 2).


## 1. Setup

In [1]:
#%pip uninstall -y mistralai
#%pip install --no-cache-dir mistralai

In [2]:
#%pip install python-dotenv

In [3]:
#%pip install pydantic
#%pip install google-genai
#%pip install mistralai

In [4]:
# 1. Force Jupyter to automatically reload external python files if you save changes to them
%load_ext autoreload
%autoreload 2

import sys, os, json, sqlite3
sys.path.insert(0, os.path.abspath(".."))

# 2. Add override=True so it forces updates if you change the .env file mid-session
from dotenv import load_dotenv
load_dotenv(override=True)  

from agent.models import DataProfile, DomainKPIResponse, KPISpec, DomainHypothesis
from agent.domain_kpi import (
    _profile_to_prompt_context,
    _validate_sql_columns,
    SQLITE_CONSTRAINTS,
    VALID_CHART_TYPES,
)

# Set the provider you want to test. 
# It will now prioritize the 'LLM_PROVIDER' value set in your .env file,
# and fall back to "gemini" if it's completely missing.
os.environ["LLM_PROVIDER"] = os.environ.get("LLM_PROVIDER", "gemini")
print("Testing provider:", os.environ["LLM_PROVIDER"])


Testing provider: mistral


In [5]:
# Load the profile you want to test against.
# Point this at any *_profile.json produced by the Profiler node.
PROFILE_PATH = "../data/profiles/online_retail_profile.json"

with open(PROFILE_PATH) as f:
    raw = json.load(f)
profile = DataProfile(**raw)
print(f"Loaded profile: {profile.row_count} rows, {profile.column_count} columns")
print(list(profile.columns.keys()))


Loaded profile: 541909 rows, 8 columns
['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']


## 2. Prompt assembly (edit this section to iterate)

This section is a **live, editable copy** of the prompt-building logic in `agent/domain_kpi.py`.
Tweak `SYSTEM_PROMPT_DRAFT` and `SQLITE_CONSTRAINTS_DRAFT` freely here — nothing you change in this
notebook affects the real pipeline until you copy it back.

The prompt has three moving parts, matching the issue's requirements:
1. **DataProfile context** — metadata only (no sample values, no min/max — those are data, not metadata)
2. **SQLite dialect constraints** — a short bounded list, not full documentation
3. **Chart type enum + worked example** — keeps the model from inventing new chart types or wandering off-format


In [6]:
# Start from the current production constraints (imported above) so you're
# iterating on top of the real baseline, not from scratch.
SQLITE_CONSTRAINTS_DRAFT = SQLITE_CONSTRAINTS

# Edit this if you want to test loosening/tightening the dialect rules:
# SQLITE_CONSTRAINTS_DRAFT = SQLITE_CONSTRAINTS + "\n- Additional constraint you're testing...\n"

CHART_TYPES_DRAFT = VALID_CHART_TYPES  # ("card", "barChart", "lineChart", "table")

SYSTEM_PROMPT_DRAFT = f"""You are a senior BI analyst. You will be given a dataset's \
schema profile — column names, inferred types, null rates, and cardinality \
only, with no actual data values. From this metadata alone, you must:

1. Infer the dataset's business domain (e.g. "e-commerce retail", \
"HR/payroll", "financial transactions").
2. Propose exactly 5 KPIs that are analytically meaningful for that domain \
and computable as SQL against the given columns.

Constraints on the SQL you generate:
{SQLITE_CONSTRAINTS_DRAFT}

Each KPI must specify a chart_type, which must be exactly one of:
{", ".join(CHART_TYPES_DRAFT)}
- "card": a single summary number (e.g. total revenue, average order value)
- "barChart": comparison across a categorical dimension
- "lineChart": a trend over time
- "table": a ranked or multi-column breakdown

Use ONLY the column names given in the profile. Do not invent columns.

Worked example (abbreviated):
Given a profile with table "data" and columns InvoiceNo (string), \
Quantity (integer), UnitPrice (float), InvoiceDate (string, datetime-like), \
Country (string) — a reasonable KPI is:
  name: "Total Revenue by Country"
  description: "Sum of (Quantity * UnitPrice) grouped by Country, \
identifying top revenue markets."
  sql_query: "SELECT Country, SUM(Quantity * UnitPrice) AS revenue FROM data \
GROUP BY Country ORDER BY revenue DESC"
  chart_type: "barChart"
  reasoning: "Revenue by market is a standard retail KPI and Country is a \
low-cardinality categorical column suited to a bar chart."

Respond only via the structured schema provided. Do not include any text \
outside the structured response.
"""

print(SYSTEM_PROMPT_DRAFT)


You are a senior BI analyst. You will be given a dataset's schema profile — column names, inferred types, null rates, and cardinality only, with no actual data values. From this metadata alone, you must:

1. Infer the dataset's business domain (e.g. "e-commerce retail", "HR/payroll", "financial transactions").
2. Propose exactly 5 KPIs that are analytically meaningful for that domain and computable as SQL against the given columns.

Constraints on the SQL you generate:
- Target dialect is SQLite (Python's sqlite3). Only use syntax SQLite supports.
- Use LIMIT, not TOP, for row limiting.
- There is no BOOLEAN type; treat 0/1 or NULL checks as needed.
- Date/time columns are stored as TEXT. Use strftime(), date(), or datetime() for any date math, not dialect-specific date functions.
- No stored procedures, no window function fallback assumptions beyond SQLite 3.25+ (RANK, ROW_NUMBER, OVER are fine).
- Use double quotes only for identifiers if a column name requires escaping; use single q

In [7]:
# Metadata-only context for the loaded profile — same function the real node uses.
context = _profile_to_prompt_context(profile)
user_content = f"Dataset profile:\n\n{context}\n\nPropose the domain hypothesis and 5 KPIs now."
print(user_content)


Dataset profile:

source_table: data
row_count: 541909
column_count: 8
columns:
  - name: InvoiceNo | dtype: string | null_rate: 0.0000 | distinct_count: 25900 | is_all_null: False
  - name: StockCode | dtype: string | null_rate: 0.0000 | distinct_count: 4070 | is_all_null: False
  - name: Description | dtype: string | null_rate: 0.0027 | distinct_count: 4223 | is_all_null: False
  - name: Quantity | dtype: integer | null_rate: 0.0000 | distinct_count: 722 | is_all_null: False
  - name: InvoiceDate | dtype: string | null_rate: 0.0000 | distinct_count: 23260 | is_all_null: False
  - name: UnitPrice | dtype: float | null_rate: 0.0000 | distinct_count: 1630 | is_all_null: False
  - name: CustomerID | dtype: float | null_rate: 0.2493 | distinct_count: 4372 | is_all_null: False
  - name: Country | dtype: string | null_rate: 0.0000 | distinct_count: 38 | is_all_null: False

Propose the domain hypothesis and 5 KPIs now.


In [8]:
# Sanity check: confirm no data values leaked into the context, only metadata.
# (Same guarantee tests/test_domain_kpi.py enforces — worth re-checking here
# any time you change _profile_to_prompt_context.)
#
# Only check STRING-typed sample values: short numeric samples (e.g. Quantity=6)
# can coincidentally appear as substrings of row_count/distinct_count figures
# elsewhere in the context, which is not a real leak. String samples (product
# descriptions, free-text fields) are the ones that actually carry distinguishing
# content and are what this check cares about.
leaks = []
for name, col in profile.columns.items():
    if col.inferred_dtype != "string":
        continue
    for sv in col.sample_values:
        if len(str(sv)) > 3 and str(sv) in context:
            leaks.append((name, sv))

assert not leaks, f"Sample values leaked into prompt context: {leaks}"
print("OK: no string sample values present in assembled context")


OK: no string sample values present in assembled context


## 3. Live LLM call

Calls the real Gemini or Mistral API (whichever `LLM_PROVIDER` is set to) using the exact
structured-output pattern from `agent/domain_kpi.py`. Requires `GEMINI_API_KEY` or
`MISTRAL_API_KEY` set in your environment / `.env`.


In [9]:
def call_gemini(system_prompt: str, user_content: str) -> DomainKPIResponse:
    from google import genai
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    model_name = os.environ.get("GEMINI_MODEL", "gemini-3.5-flash")
    response = client.models.generate_content(
        model=model_name,
        contents=user_content,
        config={
            "system_instruction": system_prompt,
            "response_mime_type": "application/json",
            "response_schema": DomainKPIResponse,
        },
    )
    parsed = getattr(response, "parsed", None)
    if parsed is not None:
        return parsed, response
    return DomainKPIResponse.model_validate_json(response.text), response


def call_mistral(system_prompt: str, user_content: str) -> DomainKPIResponse:
    from mistralai import Mistral
    client = Mistral(api_key=os.environ["MISTRAL_API_KEY"])
    model_name = os.environ.get("MISTRAL_MODEL", "mistral-large-latest")
    chat_response = client.chat.parse(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ],
        response_format=DomainKPIResponse,
        temperature=0,
    )
    parsed = chat_response.choices[0].message.parsed
    if parsed is not None:
        return parsed, chat_response
    return DomainKPIResponse.model_validate_json(chat_response.choices[0].message.content), chat_response


In [10]:
provider = os.environ["LLM_PROVIDER"]

if provider == "gemini":
    result, raw_response = call_gemini(SYSTEM_PROMPT_DRAFT, user_content)
elif provider == "mistral":
    result, raw_response = call_mistral(SYSTEM_PROMPT_DRAFT, user_content)
else:
    raise ValueError(f"Unsupported provider: {provider}")

print(f"--- Domain Hypothesis ({provider}) ---")
print(f"Domain: {result.domain_hypothesis.domain}")
print(f"Confidence: {result.domain_hypothesis.confidence}")
print(f"Reasoning: {result.domain_hypothesis.reasoning}")
print(f"\n--- {len(result.kpi_specs)} KPIs proposed ---")
for i, kpi in enumerate(result.kpi_specs, 1):
    print(f"\n[{i}] {kpi.name}  (chart_type={kpi.chart_type})")
    print(f"    {kpi.description}")
    print(f"    SQL: {kpi.sql_query}")
    print(f"    Why: {kpi.reasoning}")


ImportError: cannot import name 'Mistral' from 'mistralai' (unknown location)

## 4. Structural validation

Same checks the pipeline enforces automatically — run them here so a bad prompt version
fails visibly *before* you move on to executing SQL.


In [ ]:
# Exactly 5 KPIs, valid chart_type, SELECT-only: these are already enforced by
# DomainKPIResponse/KPISpec at parse time (an exception above would mean the LLM's
# raw output failed validation). This cell reports on what the parsed object shows.
print(f"KPI count: {len(result.kpi_specs)} (must be 5)")
print(f"Chart types used: {[k.chart_type for k in result.kpi_specs]}")
assert all(k.chart_type in VALID_CHART_TYPES for k in result.kpi_specs)
print("All chart_types valid.")


KPI count: 5 (must be 5)
Chart types used: ['card', 'barChart', 'lineChart', 'table', 'card']
All chart_types valid.


In [ ]:
# Column-hallucination check — the same deterministic validator run_domain_kpi uses.
violations = _validate_sql_columns(result.kpi_specs, profile)
if violations:
    print("VIOLATIONS FOUND:")
    for v in violations:
        print(" -", v)
else:
    print("No column violations — every sql_query references only real columns.")


VIOLATIONS FOUND:
 - KPI 'Monthly Sales Trend': sql_query references unrecognized identifier(s) ['Y', 'm'] not present in DataProfile columns ['Country', 'CustomerID', 'Description', 'InvoiceDate', 'InvoiceNo', 'Quantity', 'StockCode', 'UnitPrice'].


## 5. Execute the generated SQL for real

This is the step that tells you about *quality*, not just validity — a query can pass the
column check and still be semantically wrong (bad join logic, an aggregation that returns
one meaningless row, etc).

Two ways to feed this section real data:

- **(a) You have the actual CSV/SQLite locally** — point `DATA_SOURCE` at it and it loads real rows.
- **(b) You only have the profile JSON** (current situation for `online_retail`) — the notebook
  synthesizes an empty table with the right schema (correct column names + types, zero rows) so you
  can at least confirm the SQL **executes without error**. You won't get row-level result quality
  signal from synthetic data — for that, get the real CSV into `data/raw/` and re-point `DATA_SOURCE`.


In [ ]:
DATA_SOURCE = "../data/raw/online_retail.csv"  # change per dataset; None to force synthetic schema

conn = sqlite3.connect(":memory:")

_dtype_to_sqlite = {
    "integer": "INTEGER",
    "float": "REAL",
    "string": "TEXT",
    "boolean": "INTEGER",
    "datetime": "TEXT",
    "unknown": "TEXT",
}

if DATA_SOURCE and os.path.exists(DATA_SOURCE):
    import pandas as pd
    print(f"Loading real data from {DATA_SOURCE}")
    if DATA_SOURCE.endswith(".csv"):
        df = pd.read_csv(DATA_SOURCE, encoding="latin1")  # online_retail.csv is latin1-encoded
    else:
        raise ValueError("Add a loader branch for this file type")
    df.to_sql("data", conn, index=False, if_exists="replace")
    print(f"Loaded {len(df)} real rows into in-memory SQLite table 'data'.")
else:
    print("Real data source not found — creating an EMPTY table with the profile's schema.")
    print("SQL will be checked for EXECUTABILITY only, not result quality.")
    col_defs = ", ".join(
        f'"{name}" {_dtype_to_sqlite.get(col.inferred_dtype, "TEXT")}'
        for name, col in profile.columns.items()
    )
    conn.execute(f"CREATE TABLE data ({col_defs})")
    print("Created empty table 'data' with columns:", list(profile.columns.keys()))


Loading real data from ../data/raw/online_retail.csv
Loaded 541909 real rows into in-memory SQLite table 'data'.


In [ ]:
# Run every proposed KPI's SQL against the table and report pass/fail.
# This mirrors what the (future) SQL Executor node will do — catching this here,
# before that node exists, is exactly the value of this notebook.
results = []
for kpi in result.kpi_specs:
    try:
        cur = conn.execute(kpi.sql_query)
        rows = cur.fetchall()
        col_names = [d[0] for d in cur.description]
        results.append({"kpi": kpi.name, "status": "OK", "columns": col_names, "row_count": len(rows), "sample_rows": rows[:3]})
    except Exception as e:
        results.append({"kpi": kpi.name, "status": "FAILED", "error": str(e)})

for r in results:
    print(f"\n[{r['status']}] {r['kpi']}")
    if r["status"] == "OK":
        print(f"    columns: {r['columns']}, rows returned: {r['row_count']}")
        if r["sample_rows"]:
            print(f"    sample: {r['sample_rows']}")
    else:
        print(f"    error: {r['error']}")

n_failed = sum(1 for r in results if r["status"] == "FAILED")
print(f"\n{'ALL 5 QUERIES EXECUTED SUCCESSFULLY' if n_failed == 0 else f'{n_failed}/5 QUERIES FAILED — see above'}")



[OK] Total Revenue
    columns: ['total_revenue'], rows returned: 1
    sample: [(9747747.93400317,)]

[OK] Revenue by Country
    columns: ['Country', 'revenue'], rows returned: 10
    sample: [('United Kingdom', 8187806.364001113), ('Netherlands', 284661.54000000015), ('EIRE', 263276.81999999826)]

[OK] Monthly Sales Trend
    columns: ['sales_month', 'monthly_revenue'], rows returned: 13
    sample: [('2010-12', 748957.0199999774), ('2011-01', 560000.2600000234), ('2011-02', 498062.6500000268)]

[OK] Top 10 Selling Products
    columns: ['StockCode', 'Description', 'quantity_sold'], rows returned: 10
    sample: [('84077', 'WORLD WAR 2 GLIDERS ASSTD DESIGNS', 53847), ('85099B', 'JUMBO BAG RED RETROSPOT', 47363), ('84879', 'ASSORTED COLOUR BIRD ORNAMENT', 36381)]

[OK] Average Order Value (AOV)
    columns: ['average_order_value'], rows returned: 1
    sample: [(376.3609240927865,)]

ALL 5 QUERIES EXECUTED SUCCESSFULLY


## 6. Judgment call: does the constraint list need expanding?

If a query failed in Section 5 with a **SQLite syntax error** (not a column-hallucination —
that's already caught in Section 4), that's a signal the constraint list is missing something.
Common culprits to check for here:

- Model used a non-SQLite date function (`DATEDIFF`, `EXTRACT`, etc.) → add an explicit `strftime` example
- Model used `TOP N` instead of `LIMIT` → constraint already covers this; if it still happens, move it earlier in the list or make it a worked-example line instead of a bullet
- Model referenced a second implied table → tighten the "assume table name is 'data'" line
- Window function syntax issues → SQLite's `RANK()/ROW_NUMBER()` need `OVER (...)`; if the model omits `OVER`, add a worked example showing it

Use the cell below to log what broke and iterate — edit Section 2, re-run from there, and
compare `results` across runs.


In [ ]:
# Scratch cell — log observations here as you iterate on the prompt.
# e.g.:
# run_1_notes = "Constraint list as-is: all 5 queries pass on retail. Testing Chinook next."
# run_2_notes = "Chinook: model used julianday() unprompted — fine, SQLite supports it. No changes needed."

# DO NOT RUN THIS CELL AGAIN - STORES ALL EXPERIMENT NOTES AS LIST
notes = []


In [ ]:
run_note = "flash 3.5: violation was raised but all the sql queries ran successfully with prompt"
run_info_all = (run_note,result,raw_response)

notes.append(run_info_all)
print(notes)

[('flash 3.5: violation was raised but all the sql queries ran successfully with prompt', DomainKPIResponse(domain_hypothesis=DomainHypothesis(domain='e-commerce retail', confidence=1.0, reasoning="The columns including InvoiceNo, StockCode, Quantity, UnitPrice, CustomerID, and Country are definitive of transactional retail sales data, typically representing an e-commerce store's ledger."), kpi_specs=[KPISpec(name='Total Revenue', description='Calculates the overall revenue generated by multiplying Quantity and UnitPrice across all transactions.', sql_query='SELECT SUM(Quantity * UnitPrice) AS total_revenue FROM data', chart_type='card', reasoning='A foundational e-commerce metric representing total sales, best represented as a single high-level summary card.'), KPISpec(name='Revenue by Country', description="Aggregates total revenue grouped by the customer's country to find top geographic markets.", sql_query='SELECT Country, SUM(Quantity * UnitPrice) AS revenue FROM data GROUP BY Cou

## 7. Repeat for the other datasets

Change `PROFILE_PATH` in Section 1 and `DATA_SOURCE` in Section 5 to `chinook_profile.json` /
`hr_profile.json` (once those exist), then re-run the whole notebook top to bottom. 

Once a prompt version passes cleanly (5/5 executable, 0 column violations, sensible chart types)
across **all three** datasets, port `SYSTEM_PROMPT_DRAFT` and `SQLITE_CONSTRAINTS_DRAFT` back into
`agent/domain_kpi.py`'s `SYSTEM_PROMPT` / `SQLITE_CONSTRAINTS` constants.
